# Format and research-structure validation

Validate documented formats, participant/run relationships, and GDF headers and annotations without loading signal arrays.

Documentation authority: [Zenodo record](https://zenodo.org/records/8089820), [Scientific Data descriptor](https://www.nature.com/articles/s41597-023-02445-z), and the publisher documentation retained through `data/raw/`. Unknown semantics always force preserve-only behavior.

## Paths and safety guards

The project is a sibling of the untouched `BCI Database/`. `data/raw/` is a read-only relative symlink to that source, while `data/processed/` is the derived APFS clone.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {"notebooks", "data"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
required_directories = ("data", "models", "notebooks", "results")
if not all((PROJECT_ROOT / name).is_dir() for name in required_directories):
    raise RuntimeError("Run this notebook from bci_cleaning/ or its data/ or notebooks/ directory")

RAW_PATH = PROJECT_ROOT / "data" / "raw"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed"
REPORTS_PATH = PROJECT_ROOT / "results" / "reports"
LOGS_PATH = PROJECT_ROOT / "results" / "logs"
EXPECTED_RAW = PROJECT_ROOT.parent / "BCI Database"

if not RAW_PATH.is_symlink() or RAW_PATH.resolve(strict=True) != EXPECTED_RAW.resolve(strict=True):
    raise RuntimeError("data/raw must remain a relative symlink to the sibling BCI Database directory")

os.chdir(PROJECT_ROOT)
support_path = str(PROJECT_ROOT / "support")
if support_path not in sys.path:
    sys.path.insert(0, support_path)


## Phase implementation

In [2]:
"""Validate formats, identifiers, recordings, runs, trials, and file relationships."""

from __future__ import annotations

import collections
import csv
import os
import sys
import warnings
import zipfile
from pathlib import Path

from bci_core import (
    ARTICLE_URL,
    ZENODO_RECORD,
    Issue,
    archive_comparison_passed,
    common_parser,
    expected_participants,
    is_administrative_artifact,
    iter_files,
    log_event,
    merge_issues,
    new_run_id,
    participant_id_from_path,
    resolve_raw,
    run_id_from_name,
    validate_gdf_signature,
    validate_pdf,
    validate_python,
    validate_xml,
    validate_zip_container,
)


def _issue(
    severity: str,
    relative: str,
    participant_run: str,
    check: str,
    observed: object,
    expected: str,
    resolution: str,
    source: str = ARTICLE_URL,
) -> Issue:
    return Issue(
        severity,
        "validation",
        relative,
        participant_run,
        check,
        str(observed),
        expected,
        source,
        resolution,
    )


def _validate_csv(path: Path) -> None:
    payload = path.read_bytes()
    if path.name == "Perfomances.csv":
        text, delimiter = payload.decode("cp1252"), ";"
    else:
        text, delimiter = payload.decode("utf-8"), ","
    list(csv.reader(text.splitlines(keepends=True), delimiter=delimiter))


def _validate_gdf(path: Path, relative: str) -> tuple[dict[str, object], list[Issue]]:
    try:
        import mne
    except ImportError as exc:
        raise RuntimeError("mne is required for read-only GDF validation") from exc

    validate_gdf_signature(path)
    issues: list[Issue] = []
    participant = participant_id_from_path(relative)
    run = run_id_from_name(path.name)
    label = f"{participant}/{run}" if run else participant
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw = mne.io.read_raw_gdf(path, preload=False, verbose="ERROR")
    try:
        sfreq = float(raw.info["sfreq"])
        nchan = int(raw.info["nchan"])
        duration = float(raw.n_times / sfreq) if sfreq else 0.0
        descriptions = collections.Counter(str(value) for value in raw.annotations.description)
        if sfreq != 512.0:
            issues.append(
                _issue(
                    "error", relative, label, "sampling_rate_hz", sfreq, "512 Hz",
                    "Preserve the file and determine whether the archive or documentation explains the discrepancy.",
                )
            )
        if nchan != 32:
            issues.append(
                _issue(
                    "error", relative, label, "channel_count", nchan,
                    "32 physiological channels: 27 EEG, 3 EOG, and 2 EMG",
                    "Preserve the file and investigate against the source documentation.",
                )
            )
        if duration <= 0:
            issues.append(
                _issue(
                    "error", relative, label, "recording_duration", duration,
                    "Positive recording duration", "Restore the archive member; do not repair binary data.",
                )
            )
        if run.startswith("R"):
            trials = descriptions.get("768", 0)
            left = descriptions.get("769", 0)
            right = descriptions.get("770", 0)
            if trials != 40:
                issues.append(
                    _issue(
                        "warning", relative, label, "trial_start_count", trials,
                        "40 trials per MI run", "Preserve the recording and document the discrepancy.",
                    )
                )
            if left != 20 or right != 20:
                issues.append(
                    _issue(
                        "warning", relative, label, "cue_balance", f"left={left}; right={right}",
                        "20 left-hand and 20 right-hand cues per MI run",
                        "Preserve the recording and document the discrepancy.",
                    )
                )
            if participant == "A1" and run in {"R1", "R2"}:
                if descriptions.get("800", 0) >= 40 or descriptions.get("32770", 0) > 0:
                    issues.append(
                        _issue(
                            "info", relative, label, "documented_A1_trigger_exception",
                            f"end_trial={descriptions.get('800', 0)}; end_run={descriptions.get('32770', 0)}",
                            "A1 R1/R2 are documented as lacking end-of-trial and end-of-run triggers",
                            "No change; retain the publisher-supplied recording.",
                        )
                    )
            elif descriptions.get("800", 0) != 40:
                issues.append(
                    _issue(
                        "warning", relative, label, "trial_end_count", descriptions.get("800", 0),
                        "40 end-of-trial triggers", "Preserve and document; do not fabricate triggers.",
                    )
                )
        return (
            {
                "participant": participant,
                "run": run,
                "sfreq": sfreq,
                "nchan": nchan,
                "duration_seconds": duration,
                "annotation_count": sum(descriptions.values()),
            },
            issues,
        )
    finally:
        del raw


def _validate_participant_layout(raw: Path) -> list[Issue]:
    issues: list[Issue] = []
    signals = raw / "Signals"
    observed_dirs: dict[str, Path] = {}
    for group in ("DATA A", "DATA B", "DATA C"):
        group_path = signals / group
        if not group_path.is_dir():
            issues.append(
                _issue(
                    "error", f"Signals/{group}", "", "dataset_group_directory", "missing",
                    "Documented dataset group must exist", "Restore the group from the Zenodo archive.",
                )
            )
            continue
        for path in sorted(group_path.iterdir()):
            if path.is_dir():
                observed_dirs[path.name] = path
    expected = expected_participants()
    for participant in sorted(expected - set(observed_dirs)):
        issues.append(
            _issue(
                "error", "Signals", participant, "participant_directory", "missing",
                "Every documented participant ID must have one directory",
                "Restore the directory from the archive.",
            )
        )
    for participant in sorted(set(observed_dirs) - expected):
        issues.append(
            _issue(
                "error", observed_dirs[participant].relative_to(raw).as_posix(), participant,
                "participant_identifier", "unexpected", "A1-A60, B61-B81, or C82-C87",
                "Preserve and investigate against the archive; do not rename by inference.",
            )
        )
    expected_runs = {"OE", "CE", "R1", "R2", "R3", "R4", "R5", "R6"}
    for participant, directory in sorted(observed_dirs.items()):
        gdfs = sorted(directory.glob("*.gdf"))
        runs = {run_id_from_name(path.name) for path in gdfs}
        participant_expected = expected_runs - ({"R5", "R6"} if participant == "A59" else set())
        missing = participant_expected - runs
        extra = runs - participant_expected
        if missing:
            issues.append(
                _issue(
                    "error", directory.relative_to(raw).as_posix(), participant, "recording_set",
                    f"missing={','.join(sorted(missing))}",
                    "Two baselines and R1-R6, except documented A59 R5/R6 absence",
                    "Restore missing archive members or document the source exception.",
                )
            )
        if extra:
            issues.append(
                _issue(
                    "error", directory.relative_to(raw).as_posix(), participant, "recording_set",
                    f"extra={','.join(sorted(extra))}", "No undocumented runs",
                    "Preserve and investigate against the archive.",
                )
            )
        for gdf in gdfs:
            if not gdf.name.startswith(participant + "_"):
                issues.append(
                    _issue(
                        "error", gdf.relative_to(raw).as_posix(), participant, "filename_identifier",
                        gdf.name, f"Filename prefix {participant}_",
                        "Do not rename; verify against the archive and documentation.",
                    )
                )
        configs = [path for path in directory.iterdir() if path.suffix.lower() in {".xml", ".cfg"}]
        if participant == "C83" and not configs:
            issues.append(
                _issue(
                    "warning", directory.relative_to(raw).as_posix(), participant,
                    "configuration_files", "none", "Configuration files are normally supplied",
                    "Confirm archive membership; preserve as an unresolved source omission.",
                )
            )
        elif participant == "A59" and not configs:
            issues.append(
                _issue(
                    "info", directory.relative_to(raw).as_posix(), participant,
                    "configuration_files", "none", "A59 has a local note stating no server filter",
                    "No change; preserve the note and document the exception.",
                    source="Signals/DATA A/A59/A59=S12 pas de filtre dans serveur.txt",
                )
            )
    return issues


In [3]:
def main() -> int:
    parser = common_parser(__doc__ or "Validate dataset")
    args = parser.parse_args()
    raw = resolve_raw(args.raw)
    args.reports.mkdir(parents=True, exist_ok=True)
    args.logs.mkdir(parents=True, exist_ok=True)
    matplotlib_cache = args.logs / "matplotlib"
    matplotlib_cache.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_cache))
    run_id = new_run_id()
    issues = _validate_participant_layout(raw)
    log_event(args.logs, run_id, "validation", "start", raw=str(raw))

    if not archive_comparison_passed(args.reports / "archive_comparison.csv"):
        issues.append(
            _issue(
                "fatal", "", "", "archive_comparison", "missing or failed",
                "All local paths, sizes, and CRC32 values must match Zenodo",
                "Run 01_inventory.ipynb with network access and resolve all mismatches.",
                source=ZENODO_RECORD,
            )
        )

    gdf_count = 0
    for index, path in enumerate(iter_files(raw), start=1):
        relative = path.relative_to(raw).as_posix()
        suffix = path.suffix.lower()
        try:
            if is_administrative_artifact(relative, path):
                issues.append(
                    _issue(
                        "info", relative, "", "administrative_artifact", "publisher-supplied",
                        "Administrative metadata is not research data",
                        "Retain in raw and omit from cleaned with a manifest entry.",
                        source=ZENODO_RECORD,
                    )
                )
            elif suffix == ".gdf":
                _, file_issues = _validate_gdf(path, relative)
                issues.extend(file_issues)
                gdf_count += 1
                if gdf_count % 10 == 0:
                    log_event(args.logs, run_id, "validation", "gdf_progress", completed=gdf_count)
            elif suffix == ".xml":
                validate_xml(path)
            elif suffix in {".xlsx", ".docx"}:
                validate_zip_container(path)
            elif suffix == ".pdf":
                page_count = validate_pdf(path)
                if page_count == 0:
                    raise RuntimeError("PDF contains no pages")
            elif suffix == ".csv":
                _validate_csv(path)
            elif suffix == ".py":
                validate_python(path)
            else:
                with path.open("rb") as handle:
                    handle.read(1)
        except Exception as exc:
            issues.append(
                _issue(
                    "error", relative, participant_id_from_path(relative), "format_integrity",
                    f"{type(exc).__name__}: {exc}", "File must be readable by its documented format",
                    "Restore the exact archive member; do not repair or reinterpret scientific content.",
                    source=ZENODO_RECORD,
                )
            )

    merge_issues(args.reports / "validation_issues.csv", issues, {"validation"})
    blocking = sum(issue.severity in {"fatal", "error"} for issue in issues)
    log_event(
        args.logs, run_id, "validation", "complete", gdf_files=gdf_count,
        issues=len(issues), blocking_issues=blocking,
    )
    print(f"Validation complete: {gdf_count} GDF files; {len(issues)} issues; {blocking} blocking")
    return 0 if blocking == 0 else 2


## Execute phase

In [4]:
_arguments = [
    "03_validate.ipynb",
    "--raw", str(RAW_PATH),
    "--cleaned", str(CLEANED_PATH),
    "--reports", str(REPORTS_PATH),
    "--logs", str(LOGS_PATH),
]

_original_argv = sys.argv[:]
try:
    sys.argv = _arguments
    _exit_code = main()
finally:
    sys.argv = _original_argv

if _exit_code != 0:
    raise RuntimeError(f"Phase returned nonzero status {_exit_code}; inspect validation_issues.csv")


Validation complete: 694 GDF files; 12 issues; 0 blocking


## Privacy-safe report check

In [5]:
import csv

report_path = REPORTS_PATH / "validation_issues.csv"
with report_path.open("r", encoding="utf-8", newline="") as handle:
    report_rows = list(csv.DictReader(handle))

{
    "report": str(report_path.relative_to(PROJECT_ROOT)),
    "rows": len(report_rows),
}


{'report': 'results/reports/validation_issues.csv', 'rows': 14}